## Feature Engineering

In [1]:
import pandas as pd
import numpy as np
import ast

# Load the cleaned dataset from the previous step
df = pd.read_csv('../data/game_data_cleaned.csv')

In [2]:
# Step 1 — Convert string-lists to real lists and extract the first element
# We use .apply(ast.literal_eval) to turn "['Dev']" into ['Dev']
df['primary_developer'] = df['developers'].apply(lambda x: ast.literal_eval(x)[0] if pd.notna(x) and x != '[]' else 'Unknown')

# Step 2 — Count games per primary developer
dev_counts = df['primary_developer'].value_counts().to_dict()

# Step 3 — Map counts back
df['dev_game_count'] = df['primary_developer'].map(dev_counts)

In [4]:
# List of columns where you want to remove extreme outliers
numerical_cols = ['price', 'dlc_count', 'average_playtime_forever', 'dev_game_count']

def remove_outliers_iqr(df, columns):
    df_final = df.copy()
    for col in columns:
        if col in df_final.columns:
            Q1 = df_final[col].quantile(0.25)
            Q3 = df_final[col].quantile(0.75)
            IQR = Q3 - Q1
            
            # Define lower and upper bounds
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Filter the dataframe
            initial_count = len(df_final)
            df_final = df_final[(df_final[col] >= lower_bound) & (df_final[col] <= upper_bound)]
            
            print(f"Removed {initial_count - len(df_final)} outliers from {col}")
            
    return df_final

# Apply the function
df_cleaned = remove_outliers_iqr(df, numerical_cols)

print(f"\nFinal shape after outlier removal: {df_cleaned.shape}")

Removed 4146 outliers from price
Removed 12549 outliers from dlc_count
Removed 4232 outliers from average_playtime_forever
Removed 9811 outliers from dev_game_count

Final shape after outlier removal: (58880, 81)


In [7]:
# List of columns to remove
cols_to_drop = [
    'detailed_description', 'about_the_game', 'short_description', 'reviews', 
    'header_image', 'website', 'support_url', 'support_email', 'metacritic_url', 
    'metacritic_score', 'achievements', 'notes', 'supported_languages', 
    'full_audio_languages', 'packages', 'developers', 'publishers', 
    'categories', 'screenshots', 'movies', 'user_score', 'score_rank', 
    'discount', 'tags', 'appid', 'primary_developer'
]

# Drop the columns (errors='ignore' ensures it won't crash if a column was already dropped)
df_final = df_cleaned.drop(columns=cols_to_drop, errors='ignore')

# Check the remaining columns
print(f"Remaining columns: {df_final.columns.tolist()}")
df_final.head()


Remaining columns: ['name', 'release_date', 'required_age', 'price', 'dlc_count', 'windows', 'mac', 'linux', 'recommendations', 'positive', 'negative', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'peak_ccu', 'pct_pos_total', 'num_reviews_total', 'pct_pos_recent', 'num_reviews_recent', 'owners_clean', 'genre_360 Video', 'genre_Accounting', 'genre_Action', 'genre_Adventure', 'genre_Animation & Modeling', 'genre_Audio Production', 'genre_Casual', 'genre_Design & Illustration', 'genre_Documentary', 'genre_Early Access', 'genre_Education', 'genre_Episodic', 'genre_Free To Play', 'genre_Game Development', 'genre_Gore', 'genre_Indie', 'genre_Massively Multiplayer', 'genre_Movie', 'genre_Nudity', 'genre_Photo Editing', 'genre_RPG', 'genre_Racing', 'genre_Sexual Content', 'genre_Short', 'genre_Simulation', 'genre_Software Training', 'genre_Sports', 'genre_Strategy', 'genre_Tutorial', 'genre_Utilities', 'genre_Video Production', 'ge

,name,release_date,required_age,price,dlc_count,windows,mac,linux,recommendations,positive,...,genre_Simulation,genre_Software Training,genre_Sports,genre_Strategy,genre_Tutorial,genre_Utilities,genre_Video Production,genre_Violent,genre_Web Publishing,dev_game_count
1,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,1,0,0,1732007,1487960,...,0,0,0,0,0,0,0,0,0,1
21,Among Us,2018-11-16,0,2.99,0,1,0,0,613605,640795,...,0,0,0,0,0,0,0,0,0,1
28,Fall Guys,2020-08-03,0,0.00,0,1,0,0,417308,385931,...,0,0,1,0,0,0,0,0,0,6
30,Rocket League®,2015-07-06,0,0.00,0,1,0,0,433535,509315,...,0,0,1,0,0,0,0,0,0,1
32,Lethal Company,2023-10-23,0,9.99,0,1,0,0,386936,466206,...,0,0,0,0,0,0,0,0,0,3


In [8]:
df.to_csv('../data/game_data_cleaned.csv', index=False)